# Phase 11 — Panel construction with controls

Adds matched never-retracted authors to the panel, so effects are measured
against a counterfactual. With `CONTROL` as the omitted reference group, each
coefficient becomes an absolute difference from matched controls. Controls carry
a pseudo-retraction year inherited from their matched treated author.

**Inputs:** `data/interim/phase08_panel_extended.csv`,
`data/interim/phase08_panel_citations.csv`,
`data/interim/phase07_control_papers.csv`,
`data/interim/phase07_control_queue.csv`,
`data/interim/phase04_papers.csv`,
`data/interim/phase06_candidates_screened.csv`

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase11_panel_controls.csv` | treated and control, publications and exit |
| `data/interim/phase11_panel_citations.csv` | the same for the citation window |

Standard errors cluster on the OpenAlex source identifier (journal) of the paper
that defines each author: the retracted paper for treated authors, the sampled
paper for controls. Both sides must carry identifiers from the same namespace,
or a treated author and a control in the same venue fall into separate clusters.
The count of clusters containing both groups is reported below.

In [1]:
import gc
import os

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

TREATED_PANEL = "data/interim/phase08_panel_extended.csv"
TREATED_CIT_PANEL = "data/interim/phase08_panel_citations.csv"
TREATED_PAPERS = "data/interim/phase04_papers.csv"

CONTROL_PAPERS = "data/interim/phase07_control_papers.csv"
CONTROL_QUEUE = "data/interim/phase07_control_queue.csv"
SCREENED = "data/interim/phase06_candidates_screened.csv"

OUT_PANEL = "data/interim/phase11_panel_controls.csv"
OUT_PANEL_CIT = "data/interim/phase11_panel_citations.csv"

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]
REF_ARM = "CONTROL"

EXT_PRE, EXT_POST = 7, 6
ANALYSIS_PRE, ANALYSIS_POST = 6, 6
EXT_MIN_YEAR, EXT_MAX_YEAR = 2000, 2025

CIT_PRE, CIT_POST = 7, 3
CIT_ANALYSIS_PRE, CIT_ANALYSIS_POST = 3, 3
CIT_MIN_YEAR, CIT_MAX_YEAR = 2012, 2023

# Matches the cap applied to treated authors at Phase 8.
MAX_WORKS_PER_AUTHOR = 3000

CHUNK_ROWS = 1_000_000

pd.set_option("display.width", 220)
os.makedirs("data/interim", exist_ok=True)

## Cluster identifiers

In [2]:
def source_id_maps():
    """author_id -> OpenAlex source_id (journal), for both groups.

    Treated authors take the journal of the retracted paper; controls take the
    journal they were sampled from.
    """
    treated = {}
    if os.path.isfile(TREATED_PAPERS):
        tp = pd.read_csv(TREATED_PAPERS,
                         usecols=lambda c: c in {"author_id",
                                                 "is_focal_retraction",
                                                 "source_id"},
                         low_memory=False)
        if "is_focal_retraction" in tp.columns:
            tp = tp[tp.is_focal_retraction.astype(bool)]
        tp = tp.dropna(subset=["source_id"]).drop_duplicates("author_id")
        treated = dict(zip(tp.author_id.astype(str), tp.source_id.astype(str)))
        del tp

    control = {}
    if os.path.isfile(SCREENED):
        sc = pd.read_csv(SCREENED, usecols=["author_id", "source_id"],
                         low_memory=False)
        sc = sc.dropna(subset=["source_id"]).drop_duplicates("author_id")
        control = dict(zip(sc.author_id.astype(str), sc.source_id.astype(str)))
        del sc

    print(f"source_id for {len(treated):,} treated, {len(control):,} controls")
    _ = gc.collect()
    return treated, control


TREATED_MAP, CONTROL_MAP = source_id_maps()

source_id for 54,961 treated, 8,118,192 controls


## Control aggregates

The control extract is read in chunks and reduced to annual publication counts
and, for the citation panel, annual citation totals, so the full extract is
never resident.

In [3]:
def control_aggregates(with_citations, chunk_rows=CHUNK_ROWS):
    """Annual publication and citation counts per control author.

    Returns (publications, citations, excluded). Duplicate author-work pairs are
    removed within chunks and again across them.
    """
    if not os.path.isfile(CONTROL_PAPERS):
        raise SystemExit(f"{CONTROL_PAPERS} not found; run Phase 7 first")

    usecols = ["author_id", "work_id", "pub_year"]
    if with_citations:
        usecols.append("counts_by_year")

    pub_parts, cit_parts, seen_counts = [], [], []
    n_rows = 0

    reader = pd.read_csv(CONTROL_PAPERS, usecols=usecols,
                         chunksize=chunk_rows, low_memory=False,
                         on_bad_lines="skip")
    for chunk in reader:
        n_rows += len(chunk)
        chunk["author_id"] = chunk.author_id.astype(str)
        chunk = chunk.drop_duplicates(subset=["author_id", "work_id"])
        chunk = chunk.dropna(subset=["pub_year"])
        chunk["pub_year"] = chunk.pub_year.astype(int)

        seen_counts.append(chunk.groupby("author_id").size())

        pub_parts.append(chunk.groupby(["author_id", "pub_year"])
                              .size().rename("publications"))

        if with_citations and "counts_by_year" in chunk.columns:
            s = chunk[["author_id", "counts_by_year"]].dropna()
            s = s[s.counts_by_year.astype(str).str.len() > 0]
            if len(s):
                # "YYYY:n|YYYY:n" expands to one row per year, summed within
                # the chunk so the expansion never outlives it.
                ex = s.assign(
                    pair=s.counts_by_year.astype(str).str.split("|")
                ).explode("pair")
                ex = ex[ex.pair.str.contains(":", na=False)]
                parts = ex.pair.str.split(":", n=1, expand=True)
                ex = ex.assign(
                    year=pd.to_numeric(parts[0], errors="coerce"),
                    n=pd.to_numeric(parts[1], errors="coerce"))
                ex = ex.dropna(subset=["year", "n"])
                ex["year"] = ex.year.astype(int)
                cit_parts.append(ex.groupby(["author_id", "year"])
                                   .n.sum().rename("citations"))
                del ex, parts, s
        del chunk
        _ = gc.collect()

    pubs = (pd.concat(pub_parts).groupby(level=[0, 1]).sum()
              .reset_index()) if pub_parts else pd.DataFrame(
                  columns=["author_id", "pub_year", "publications"])
    del pub_parts

    cits = (pd.concat(cit_parts).groupby(level=[0, 1]).sum()
              .reset_index()) if cit_parts else pd.DataFrame(
                  columns=["author_id", "year", "citations"])
    del cit_parts

    per_author = (pd.concat(seen_counts).groupby(level=0).sum()
                  if seen_counts else pd.Series(dtype=int))
    del seen_counts

    excluded = set()
    if MAX_WORKS_PER_AUTHOR is not None and len(per_author):
        over = per_author[per_author > MAX_WORKS_PER_AUTHOR]
        if len(over):
            print(f"  excluded {len(over):,} controls above "
                  f"{MAX_WORKS_PER_AUTHOR:,} works (largest {over.max():,})")
            excluded = set(over.index)
            pubs = pubs[~pubs.author_id.isin(excluded)]
            if len(cits):
                cits = cits[~cits.author_id.isin(excluded)]

    print(f"  read {n_rows:,} rows, {len(per_author):,} controls")
    _ = gc.collect()
    return pubs, cits, excluded

## Panel assembly

The author-year grid is built directly from the pseudo-retraction years by
vectorised operations, and the aggregates are merged onto it.

In [4]:
def load_treated(path, label):
    p = pd.read_csv(path, low_memory=False)
    t = p[p.balanced & p.in_primary & p.arm.isin(ARMS)].copy()
    t["author_id"] = t.author_id.astype(str)
    del p
    print(f"{label}: {len(t):,} rows, {t.author_id.nunique():,} authors")
    print("  " + t.drop_duplicates("author_id").arm.value_counts()
            .to_string().replace("\n", "\n  "))
    return t


def build_control_panel(pubs, cits, excluded, pre, post,
                        min_year, max_year, with_citations, label):
    queue = pd.read_csv(CONTROL_QUEUE)
    queue["author_id"] = queue.author_id.astype(str)
    have = set(pubs.author_id.unique())
    q = queue[queue.author_id.isin(have) &
              ~queue.author_id.isin(excluded)].copy()
    print(f"  controls with extracted works: {len(q):,} of {len(queue):,} "
          f"({len(q) / max(len(queue), 1):.1%})")
    del queue

    offsets = np.arange(-pre, post + 1)
    n_off = len(offsets)
    R = q.first_retraction_year.astype(int).to_numpy()

    grid = pd.DataFrame({
        "author_id": np.repeat(q.author_id.to_numpy(), n_off),
        "year": (R[:, None] + offsets[None, :]).ravel(),
        "event_time": np.tile(offsets, len(q)),
        "first_retraction_year": np.repeat(R, n_off),
    })
    grid = grid[(grid.year >= min_year) & (grid.year <= max_year)]

    grid = grid.merge(pubs.rename(columns={"pub_year": "year"}),
                      on=["author_id", "year"], how="left")
    grid["publications"] = grid.publications.fillna(0).astype(int)
    grid["active"] = (grid.publications > 0).astype(int)
    grid["arm"] = REF_ARM

    if with_citations:
        if len(cits):
            grid = grid.merge(cits, on=["author_id", "year"], how="left")
            grid["citations"] = grid.citations.fillna(0).astype(int)
        else:
            grid["citations"] = 0

    cov = [c for c in ["career_band", "income_group", "matched_arm"]
           if c in q.columns]
    grid = grid.merge(q[["author_id"] + cov], on="author_id", how="left")

    print(f"  {label}: {len(grid):,} rows, {grid.author_id.nunique():,} authors")
    _ = gc.collect()
    return grid

## Combine

In [5]:
def combine(treat, ctrl, a_pre, a_post):
    keep = [c for c in ["author_id", "year", "event_time", "publications",
                        "active", "citations", "arm", "first_retraction_year",
                        "career_band", "income_group", "first_position",
                        "subject_group", "lag_band"]
            if c in treat.columns and c in ctrl.columns]
    dropped = [c for c in ["career_band", "income_group"] if c not in keep]
    if dropped:
        print(f"  [!] absent on one side, dropped from the merge: {dropped}")

    panel = pd.concat([treat[keep], ctrl[keep]], ignore_index=True)

    aid = panel.author_id.astype(str)
    panel["cluster_id"] = aid.map(TREATED_MAP).fillna(aid.map(CONTROL_MAP))
    missing = panel.cluster_id.isna()
    if missing.any():
        # Singleton clusters, not one shared bucket.
        panel.loc[missing, "cluster_id"] = "UNRESOLVED_" + aid[missing]
        print(f"  {int(missing.sum()):,} rows without a source_id, given "
              f"singleton clusters")

    t_cl = set(panel.loc[panel.arm != REF_ARM, "cluster_id"])
    c_cl = set(panel.loc[panel.arm == REF_ARM, "cluster_id"])
    shared = len(t_cl & c_cl)
    print(f"  clusters: {len(t_cl):,} treated, {len(c_cl):,} control, "
          f"{shared:,} shared")
    if shared == 0:
        print("  [!] no shared clusters; check the source_id namespace in "
              "phase04_papers.csv and phase06_candidates_screened.csv")

    panel = panel[(panel.event_time >= -a_pre) & (panel.event_time <= a_post)]

    need = a_pre + a_post + 1
    span = panel.groupby("author_id").event_time.nunique()
    keep_ids = set(span[span == need].index)
    n_before = panel.author_id.nunique()
    panel = panel[panel.author_id.isin(keep_ids)]
    print(f"  dropped {n_before - len(keep_ids):,} authors not spanning "
          f"-{a_pre}..+{a_post}")
    _ = gc.collect()
    return panel

## Extended panel

In [6]:
print("\nEXTENDED PANEL")

pubs, cits, excluded = control_aggregates(with_citations=False)
ctrl_ext = build_control_panel(pubs, cits, excluded, EXT_PRE, EXT_POST,
                               EXT_MIN_YEAR, EXT_MAX_YEAR,
                               with_citations=False, label="control panel")
del pubs, cits
_ = gc.collect()

treat_ext = load_treated(TREATED_PANEL, "treated")
panel = combine(treat_ext, ctrl_ext, ANALYSIS_PRE, ANALYSIS_POST)
del treat_ext
_ = gc.collect()


EXTENDED PANEL
  excluded 8 controls above 3,000 works (largest 155,356)
  read 6,176,685 rows, 48,786 controls
  controls with extracted works: 48,778 of 48,786 (100.0%)
  control panel: 616,356 rows, 48,778 authors
treated: 242,116 rows, 17,294 authors
  arm
  AUTHOR_MISCONDUCT       9617
  HONEST_ERROR            4894
  EDITORIAL_COMPROMISE    2783
  2,226 rows without a source_id, given singleton clusters
  clusters: 2,423 treated, 2,782 control, 1,565 shared
  dropped 29,519 authors not spanning -6..+6


## The balanced sample

In [7]:
def summarise(p, ctrl, label):
    if p is None or p.empty:
        return
    print(f"\n{label}")
    a = p.drop_duplicates("author_id")
    print(a.arm.value_counts().to_string())

    n_ctrl = int((a.arm == REF_ARM).sum())
    n_treat = int((a.arm != REF_ARM).sum())
    print(f"\ncontrols per treated author: {n_ctrl / max(n_treat, 1):.2f}")
    print(f"rows                         {len(p):,}")
    print(f"calendar years               {int(p.year.min())}-{int(p.year.max())}")

    if ctrl is not None and "matched_arm" in ctrl.columns:
        mc = ctrl.drop_duplicates("author_id").matched_arm.value_counts()
        if len(mc):
            print(f"\ncontrols by the arm they were matched to")
            print("  " + mc.to_string().replace("\n", "\n  "))

    al = p.groupby(["arm", "event_time"]).year.mean().unstack()
    if REF_ARM in al.index:
        treated_mean = al.loc[[i for i in al.index if i != REF_ARM]].mean()
        offset = (treated_mean - al.loc[REF_ARM]).abs().max()
        print(f"\nlargest calendar offset between groups: {offset:.2f} years")
        if offset > 0.25:
            print("  [!] groups occupy different calendar years at some "
                  "event times")


summarise(panel, ctrl_ext, "EXTENDED PANEL, balanced")


EXTENDED PANEL, balanced
arm
CONTROL                 19259
AUTHOR_MISCONDUCT        9617
HONEST_ERROR             4894
EDITORIAL_COMPROMISE     2783

controls per treated author: 1.11
rows                         475,189
calendar years               2009-2025

controls by the arm they were matched to
  matched_arm
  AUTHOR_MISCONDUCT       25364
  HONEST_ERROR            10917
  EDITORIAL_COMPROMISE     7127
  UNCONFIRMED_CONCERNS     4302
  ETHICS_VIOLATION          723
  UNCLASSIFIED              345

largest calendar offset between groups: 0.05 years


## Raw group means

No model: the share of each group publishing at each event time.

In [8]:
m = panel.groupby(["arm", "event_time"]).active.mean().unstack()
print("share publishing, by group and event time\n")
print(m.round(3).to_string())

ref = m[list(range(-2, 0))].mean(axis=1)
print(f"\nchange from the reference period to +{ANALYSIS_POST}")
for arm in [REF_ARM] + ARMS:
    if arm in m.index:
        print(f"  {arm:<24} {m.loc[arm, ANALYSIS_POST] - ref[arm]:+.4f}")

share publishing, by group and event time

event_time               -6     -5     -4     -3     -2     -1      0      1      2      3      4      5      6
arm                                                                                                            
AUTHOR_MISCONDUCT     0.703  0.731  0.761  0.798  0.798  0.823  0.891  0.753  0.746  0.730  0.731  0.715  0.712
CONTROL               0.702  0.738  0.774  0.826  0.822  0.820  0.790  0.777  0.772  0.750  0.737  0.724  0.706
EDITORIAL_COMPROMISE  0.694  0.723  0.761  0.808  0.836  0.883  0.900  0.818  0.802  0.800  0.797  0.788  0.775
HONEST_ERROR          0.730  0.761  0.793  0.837  0.845  0.885  0.937  0.821  0.809  0.794  0.780  0.782  0.767

change from the reference period to +6
  CONTROL                  -0.1154
  AUTHOR_MISCONDUCT        -0.0984
  HONEST_ERROR             -0.0977
  EDITORIAL_COMPROMISE     -0.0841


## Write the extended panel

In [9]:
panel.to_csv(OUT_PANEL, index=False)
print(f"{OUT_PANEL}: {len(panel):,} rows, "
      f"{panel.author_id.nunique():,} authors")

del panel, ctrl_ext, m
_ = gc.collect()

data/interim/phase11_panel_controls.csv: 475,189 rows, 36,553 authors


## Citation panel

In [10]:
panel_cit = None
if os.path.isfile(TREATED_CIT_PANEL):
    print("\nCITATION PANEL")
    pubs, cits, excluded = control_aggregates(with_citations=True)
    ctrl_cit = build_control_panel(pubs, cits, excluded, CIT_PRE, CIT_POST,
                                   CIT_MIN_YEAR, CIT_MAX_YEAR,
                                   with_citations=True,
                                   label="control citation panel")
    del pubs, cits
    _ = gc.collect()

    treat_cit = load_treated(TREATED_CIT_PANEL, "treated")
    panel_cit = combine(treat_cit, ctrl_cit,
                        CIT_ANALYSIS_PRE, CIT_ANALYSIS_POST)
    del treat_cit
    _ = gc.collect()

    summarise(panel_cit, ctrl_cit, "CITATION PANEL, balanced")

    panel_cit.to_csv(OUT_PANEL_CIT, index=False)
    print(f"\n{OUT_PANEL_CIT}: {len(panel_cit):,} rows, "
          f"{panel_cit.author_id.nunique():,} authors")
    del ctrl_cit, panel_cit
    _ = gc.collect()
else:
    print(f"{TREATED_CIT_PANEL} not found; skipped")


CITATION PANEL
  excluded 8 controls above 3,000 works (largest 155,356)
  read 6,176,685 rows, 48,786 controls
  controls with extracted works: 48,778 of 48,786 (100.0%)
  control citation panel: 464,978 rows, 48,778 authors
treated: 217,456 rows, 22,633 authors
  arm
  AUTHOR_MISCONDUCT       12867
  HONEST_ERROR             6522
  EDITORIAL_COMPROMISE     3244
  1,965 rows without a source_id, given singleton clusters
  clusters: 2,945 treated, 2,782 control, 1,877 shared
  dropped 23,299 authors not spanning -3..+3

CITATION PANEL, balanced
arm
CONTROL                 25479
AUTHOR_MISCONDUCT       12867
HONEST_ERROR             6522
EDITORIAL_COMPROMISE     3244

controls per treated author: 1.13
rows                         336,784
calendar years               2012-2023

controls by the arm they were matched to
  matched_arm
  AUTHOR_MISCONDUCT       25364
  HONEST_ERROR            10917
  EDITORIAL_COMPROMISE     7127
  UNCONFIRMED_CONCERNS     4302
  ETHICS_VIOLATION          7